# LeetCode 1027 — Longest Arithmetic Subsequence bằng Python

Link đề: https://leetcode.com/problems/longest-arithmetic-subsequence/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (độ dài, luôn `≥ 2` vì `n ≥ 2` và hai phần tử bất kỳ tạo thành cấp số cộng). Điểm cốt lõi: quy hoạch động theo **(chỉ số cuối, công sai)**: `dp[j][d] = dp[i][d] + 1` khi `nums[j] − nums[i] = d`, `i < j`. Oracle độc lập: **duyệt mọi dãy con** (`2^n`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dãy con (xoá một số phần tử, giữ thứ tự) mà các hiệu liên tiếp bằng nhau; tìm độ dài lớn nhất.
2. Nắm công cụ: DP hai chiều `(j, d)` bằng `dict`, mảng theo độ lệch công sai, duyệt từng công sai với mảng theo giá trị, mở rộng tham lam từ một cặp.
3. Đi từ **thử mọi dãy con** → **DP `dict` theo (cuối, công sai)** → **DP mảng theo `d + 500`** → **mỗi công sai một lượt với mảng theo giá trị** → **mở rộng tham lam từ mỗi cặp**.
4. Hiểu mấu chốt: một cấp số cộng được xác định bởi **hai phần tử cuối và công sai**; `dp[j][d]` = độ dài dài nhất của cấp số cộng công sai `d` **kết thúc tại `j`**. Mọi cặp `(i, j)` ứng với `d = nums[j] − nums[i]` cho độ dài tối thiểu `2`.
5. Tránh lỗi kinh điển: khởi tạo `dp` sai (độ dài của cặp phải là `2`), dùng `|hiệu|` (trộn tăng và giảm), nhầm **dãy con** với **dãy con liên tiếp**, chỉ xét công sai dương, dùng chung `dp` theo `d` giữa các chỉ số (đếm sai), trả `dp` của phần tử cuối thay vì max toàn cục.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, trả về **độ dài** của **dãy con cấp số cộng** dài nhất.

Lưu ý: dãy con là mảng nhận được từ mảng gốc bằng cách xoá một số (hoặc không) phần tử mà **không đổi thứ tự** các phần tử còn lại. Dãy `seq` là cấp số cộng nếu `seq[i+1] − seq[i]` bằng nhau với mọi `i`.

**Ví dụ 1:** `nums = [3, 6, 9, 12]` → `4`.

**Ví dụ 2:** `nums = [9, 4, 7, 2, 10]` → `3` (`[4, 7, 10]`).

**Ví dụ 3:** `nums = [20, 1, 15, 3, 10, 5, 8]` → `4` (`[20, 15, 10, 5]`).

**Ràng buộc:** `2 <= nums.length <= 1000`; `0 <= nums[i] <= 500`.

In [1]:
import itertools
import random
import time
from collections import defaultdict
from typing import List

examples = [([3, 6, 9, 12], 4), ([9, 4, 7, 2, 10], 3), ([20, 1, 15, 3, 10, 5, 8], 4), ([1, 1, 1], 3), ([1, 2], 2)]
for a, want in examples:
    print(f"nums = {a} -> {want}")

nums = [3, 6, 9, 12] -> 4
nums = [9, 4, 7, 2, 10] -> 3
nums = [20, 1, 15, 3, 10, 5, 8] -> 4
nums = [1, 1, 1] -> 3
nums = [1, 2] -> 2


## 3. Kiến thức nền

### 3.1 Dãy con cấp số cộng

Một dãy chỉ số `i₁ < i₂ < … < i_k` cho cấp số cộng khi `nums[i_{t+1}] − nums[i_t]` là hằng. Dãy gồm `1` hoặc `2` phần tử luôn là cấp số cộng (mọi dãy con 2 phần tử đều đúng) nên đáp án `≥ 2`.

In [2]:
def is_arith(seq):
    return len(seq) <= 2 or all(seq[k + 1] - seq[k] == seq[1] - seq[0] for k in range(len(seq) - 1))


print(is_arith([4, 7, 10]), is_arith([4, 7, 11]), is_arith([5]), is_arith([2, 9]), is_arith([5, 5, 5]))

True False True True True


### 3.2 Trạng thái `(chỉ số cuối, công sai)`

Nếu cấp số cộng công sai `d` kết thúc tại `i` và `nums[j] − nums[i] = d` với `j > i` thì thêm `nums[j]` ta được cấp số cộng công sai `d` kết thúc tại `j`. Vậy `dp[j][d] = dp[i][d] + 1` (với `dp[i][d] = 1` mặc định tức chỉ có phần tử `i`... và cặp `(i, j)` cho độ dài `2`).

### 3.3 Phạm vi công sai

Với `0 ≤ nums[i] ≤ 500`, công sai `d ∈ [−500, 500]` (1001 giá trị) — cho phép dùng mảng thay `dict` hoặc lặp theo từng công sai.

In [3]:
nums = [9, 4, 7, 2, 10]
dp = [dict() for _ in nums]
for j in range(len(nums)):
    for i in range(j):
        d = nums[j] - nums[i]
        dp[j][d] = dp[i].get(d, 1) + 1                 # dp[i][d] chưa có: dãy chỉ gồm nums[i] (độ dài 1) -> thêm nums[j]
print(dp)
print("max =", max(max(r.values()) for r in dp if r))

[{}, {-5: 2}, {-2: 2, 3: 2}, {-7: 2, -2: 2, -5: 2}, {1: 2, 6: 2, 3: 3, 8: 2}]
max = 3


## 4. Ý tưởng nền tảng / trực giác

`nums = [9, 4, 7, 2, 10]`: cặp `(4, 7)` có `d = 3`; sau đó `10 − 7 = 3` nên `[4, 7, 10]` kéo dài lên `3`. DP ghi `dp[idx của 10][3] = dp[idx của 7][3] + 1 = 3`.

Số trạng thái `(j, d)` tối đa `n · 1001`; mỗi cặp `(i, j)` cập nhật một trạng thái ⇒ `O(n²)`.

## 5. Brute force — thử mọi dãy con

**Ý tưởng:** theo đúng định nghĩa: duyệt mọi dãy con (mặt nạ bit), kiểm tra cấp số cộng, lấy độ dài lớn nhất. `O(2^n · n)` — chỉ cho `n ≲ 14`, dùng làm **oracle**.

In [4]:
def arith_brute(nums):
    n = len(nums)
    best = 2
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if len(seq) > best and is_arith(seq):
            best = len(seq)
    return best


for a, want in examples:
    if len(a) <= 12:
        assert arith_brute(a) == want, (a, arith_brute(a))
print([arith_brute(a) for a, _ in examples])

[4, 3, 4, 3, 2]


In [5]:
# Code nộp LeetCode cho Brute force (2^n: chỉ minh hoạ)
class SolutionBrute:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n, best = len(nums), 2
        for mask in range(1, 1 << n):
            seq = [nums[i] for i in range(n) if mask >> i & 1]
            if len(seq) > best and all(seq[k + 1] - seq[k] == seq[1] - seq[0] for k in range(len(seq) - 1)):
                best = len(seq)
        return best

**Độ phức tạp:** `O(2^n · n)`.

**Nút thắt:** ta liệt kê cả dãy trong khi cấp số cộng được kéo dài **từng phần tử một** dựa vào (phần tử cuối, công sai).

## 6. Bước đột phá

`dp[j][d]` = độ dài lớn nhất của cấp số cộng công sai `d` kết thúc tại `j`. Với mỗi `i < j`: `d = nums[j] − nums[i]`, `dp[j][d] = dp[i].get(d, 1) + 1`. Đáp án là max toàn bộ. `O(n²)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DP `dict` theo (cuối, công sai) (đáp án chính), **Cách 3** DP mảng theo `d + 500`, **Cách 4** mỗi công sai một lượt với mảng theo giá trị, **Cách 5** mở rộng tham lam từ mỗi cặp.

### Cách 2 — DP `dict` theo (chỉ số cuối, công sai)

**Giải thích từng đoạn:**

1. `dp = [dict() for _ in nums]`; `ans = 2`.
2. Với `j` từ `0..n−1`, `i` từ `0..j−1`: `d = nums[j] − nums[i]`; `dp[j][d] = dp[i].get(d, 1) + 1`; `ans = max(ans, dp[j][d])`.

In [6]:
def arith_dict(nums):
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            dp[j][d] = dp[i].get(d, 1) + 1                # dp[i][d] chưa có: chỉ có nums[i] (độ dài 1)
            if dp[j][d] > ans:
                ans = dp[j][d]
    return ans


for a, want in examples:
    assert arith_dict(a) == want
print([arith_dict(a) for a, _ in examples])

[4, 3, 4, 3, 2]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionDict:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n = len(nums)
        dp = [{} for _ in range(n)]
        ans = 2
        for j in range(n):
            for i in range(j):
                d = nums[j] - nums[i]
                dp[j][d] = dp[i].get(d, 1) + 1
                ans = max(ans, dp[j][d])
        return ans

**Độ phức tạp:** `O(n²)` thời gian, `O(n · số công sai)` bộ nhớ (≤ `O(n²)`).

### Cách 3 — DP mảng theo `d + 500`

**Ý tưởng:** thay `dict` bằng danh sách `2·(max − min) + 1 ≤ 1001` phần tử mỗi chỉ số (công sai `d + OFF` với `OFF = max − min ≤ 500`), khởi tạo `1`. Nhanh hơn băm nhưng tốn `n · 1001` bộ nhớ.

In [8]:
def arith_array(nums):
    n = len(nums)
    OFF = max(nums) - min(nums)                         # |công sai| <= max - min (<= 500 theo đề)
    dp = [[1] * (2 * OFF + 1) for _ in range(n)]       # dp[j][d + OFF] = độ dài cấp số cộng công sai d kết thúc tại j (tối thiểu 1)
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i] + OFF
            if dp[i][d] + 1 > dp[j][d]:
                dp[j][d] = dp[i][d] + 1
                if dp[j][d] > ans:
                    ans = dp[j][d]
    return ans


for a, want in examples:
    assert arith_array(a) == want
print([arith_array(a) for a, _ in examples])

[4, 3, 4, 3, 2]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionArray:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n, OFF = len(nums), max(nums) - min(nums)
        dp = [[1] * (2 * OFF + 1) for _ in range(n)]
        ans = 2
        for j in range(n):
            for i in range(j):
                d = nums[j] - nums[i] + OFF
                if dp[i][d] + 1 > dp[j][d]:
                    dp[j][d] = dp[i][d] + 1
                    ans = max(ans, dp[j][d])
        return ans

**Độ phức tạp:** `O(n²)` thời gian, `O(n · 1001)` bộ nhớ.

### Cách 4 — Mỗi công sai một lượt với mảng theo giá trị

**Ý tưởng:** cố định công sai `d ∈ [−(max − min), max − min] ⊆ [−500, 500]`; duyệt `nums` theo thứ tự với mảng `best[v]` = độ dài lớn nhất của cấp số cộng công sai `d` **kết thúc bằng giá trị `v`** (đã thấy); với `x`: `prev = x − d`; nếu `prev` hợp lệ và đã có `best[prev]` thì `best[x] = max(best[x], best[prev] + 1)` ngược lại `best[x] = max(best[x], 1)`. Với `d = 0` cần cẩn thận (`prev = x`): dùng giá trị `best[prev]` **trước** khi cập nhật `best[x]`.

In [10]:
def arith_per_diff(nums):
    ans = 2
    span = max(nums) - min(nums)
    for d in range(-span, span + 1):                     # công sai chỉ có thể nằm trong [-(max-min), max-min] (<= 500)
        best = [0] * 501                                 # best[v]: độ dài lớn nhất kết thúc bằng giá trị v (0 = chưa gặp)
        for x in nums:
            prev = x - d
            cand = best[prev] + 1 if 0 <= prev <= 500 and best[prev] else 1    # đọc best[prev] TRƯỚC khi ghi best[x]
            if cand > best[x]:
                best[x] = cand
            if best[x] > ans:
                ans = best[x]
    return ans


for a, want in examples:
    assert arith_per_diff(a) == want
print([arith_per_diff(a) for a, _ in examples])

[4, 3, 4, 3, 2]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionPerDiff:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        ans = 2
        span = max(nums) - min(nums)
        for d in range(-span, span + 1):
            best = [0] * 501
            for x in nums:
                prev = x - d
                cand = best[prev] + 1 if 0 <= prev <= 500 and best[prev] else 1
                if cand > best[x]:
                    best[x] = cand
                    ans = max(ans, cand)
        return ans

**Độ phức tạp:** `O(1001 · n)` ≈ `10^6` thời gian, `O(501)` bộ nhớ — nhanh và ít bộ nhớ khi giá trị nhỏ.

### Cách 5 — Mở rộng tham lam từ mỗi cặp

**Ý tưởng:** với mỗi cặp `(i, j)`, `i < j`, `d = nums[j] − nums[i]`; mở rộng: tìm giá trị `nums[j] + d` ở chỉ số **sớm nhất** sau `j` (bằng danh sách vị trí + `bisect`), lặp. Chiều dài chuỗi thu được là dài nhất cho cặp bắt đầu `(i, j)` (chọn sớm nhất là tối ưu). `O(n² · L)` — dùng cho `n` nhỏ.

In [12]:
import bisect


def arith_extend(nums):
    n = len(nums)
    pos = defaultdict(list)
    for idx, v in enumerate(nums):
        pos[v].append(idx)
    ans = 2
    for i in range(n):
        for j in range(i + 1, n):
            d = nums[j] - nums[i]
            cur, length = j, 2
            while True:
                nxt_val = nums[cur] + d
                lst = pos.get(nxt_val)
                if not lst:
                    break
                k = bisect.bisect_right(lst, cur)          # chỉ số sớm nhất > cur có giá trị nxt_val
                if k == len(lst):
                    break
                cur = lst[k]
                length += 1
            ans = max(ans, length)
    return ans


for a, want in examples:
    assert arith_extend(a) == want
print([arith_extend(a) for a, _ in examples])

[4, 3, 4, 3, 2]


In [13]:
# Code nộp LeetCode cho Cách 5 (O(n^2·L): chỉ cho n nhỏ/vừa)
import bisect


class SolutionExtend:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n = len(nums)
        pos = {}
        for idx, v in enumerate(nums):
            pos.setdefault(v, []).append(idx)
        ans = 2
        for i in range(n):
            for j in range(i + 1, n):
                d = nums[j] - nums[i]
                cur, length = j, 2
                while True:
                    lst = pos.get(nums[cur] + d)
                    if not lst:
                        break
                    k = bisect.bisect_right(lst, cur)
                    if k == len(lst):
                        break
                    cur = lst[k]
                    length += 1
                ans = max(ans, length)
        return ans

**Độ phức tạp:** `O(n² · L · log)` xấu nhất với `L` độ dài dãy; `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `nums = [9, 4, 7, 2, 10]` (chỉ số `0..4`):

| `j` | `i` | `d = nums[j] − nums[i]` | `dp[i].get(d, 1)` | `dp[j][d]` |
|---|---|---|---|---|
| 1 (`4`) | 0 | −5 | 1 | 2 |
| 2 (`7`) | 0 | −2 | 1 | 2 |
| 2 (`7`) | 1 | 3 | 1 | 2 |
| 3 (`2`) | 0 | −7 | 1 | 2 |
| 3 (`2`) | 1 | −2 | 1 | 2 |
| 3 (`2`) | 2 | −5 | 1 | 2 |
| 4 (`10`) | 0 | 1 | 1 | 2 |
| 4 (`10`) | 1 | 6 | 1 | 2 |
| 4 (`10`) | 2 | **3** | `dp[2][3] = 2` | **3** |
| 4 (`10`) | 3 | 8 | 1 | 2 |

Lớn nhất `3` ✓ (`[4, 7, 10]`). Hàm trace in từng bước:

In [14]:
def trace_dict(nums):
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    print("nums =", nums)
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            prev = dp[i].get(d, 1)
            dp[j][d] = prev + 1
            tag = " <-- kéo dài" if prev > 1 else ""
            ans = max(ans, dp[j][d])
            print(f"  j={j} (nums={nums[j]}), i={i} (nums={nums[i]}): d={d:3d}, dp[i][d]={prev} -> dp[j][d]={dp[j][d]}{tag}")
    print("=>", ans)
    return ans


assert trace_dict([9, 4, 7, 2, 10]) == 3

nums = [9, 4, 7, 2, 10]
  j=1 (nums=4), i=0 (nums=9): d= -5, dp[i][d]=1 -> dp[j][d]=2
  j=2 (nums=7), i=0 (nums=9): d= -2, dp[i][d]=1 -> dp[j][d]=2
  j=2 (nums=7), i=1 (nums=4): d=  3, dp[i][d]=1 -> dp[j][d]=2
  j=3 (nums=2), i=0 (nums=9): d= -7, dp[i][d]=1 -> dp[j][d]=2
  j=3 (nums=2), i=1 (nums=4): d= -2, dp[i][d]=1 -> dp[j][d]=2
  j=3 (nums=2), i=2 (nums=7): d= -5, dp[i][d]=1 -> dp[j][d]=2
  j=4 (nums=10), i=0 (nums=9): d=  1, dp[i][d]=1 -> dp[j][d]=2
  j=4 (nums=10), i=1 (nums=4): d=  6, dp[i][d]=1 -> dp[j][d]=2
  j=4 (nums=10), i=2 (nums=7): d=  3, dp[i][d]=2 -> dp[j][d]=3 <-- kéo dài
  j=4 (nums=10), i=3 (nums=2): d=  8, dp[i][d]=1 -> dp[j][d]=2
=> 3


**Trace công sai `0`:** `nums = [1, 1, 1]`: `dp[1][0] = 2` (từ `i = 0`); với `j = 2`: `i = 0` cho `dp[0].get(0, 1) + 1 = 2`, rồi `i = 1` cho `dp[1][0] + 1 = 3`. Hai lần gán cùng khoá `dp[2][0]`; giá trị đúng là **max** của các ứng viên (`3`). Bản ghi đè theo `i` tăng dần cho kết quả đúng vì ứng viên của `i` lớn hơn không bao giờ nhỏ hơn ứng viên của `i` nhỏ hơn có cùng giá trị `nums[i]` (mục 10b); để không phụ thuộc lập luận này, mã nộp dùng `max` tường minh.

In [15]:
print("ghi đè:", arith_dict([1, 1, 1]), arith_dict([1, 5, 1, 1]), "| có max tường minh:", arith_array([1, 1, 1]), arith_array([1, 5, 1, 1]))
assert arith_dict([1, 1, 1]) == 3 and arith_dict([1, 5, 1, 1]) == arith_array([1, 5, 1, 1]) == arith_brute([1, 5, 1, 1]) == 3

ghi đè: 3 3 | có max tường minh: 3 3


## 9. Code nộp LeetCode

Chọn **Cách 2 với phép `max` trên `dp[j][d]`** (an toàn tuyệt đối, không phụ thuộc thứ tự). Giữ nguyên `class Solution`, `longestArithSeqLength(self, nums)`:

In [16]:
class Solution:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n = len(nums)
        dp = [{} for _ in range(n)]                    # dp[j][d]: độ dài lớn nhất của cấp số cộng công sai d kết thúc tại j
        ans = 2
        for j in range(n):
            for i in range(j):
                d = nums[j] - nums[i]
                length = dp[i].get(d, 1) + 1           # nếu chưa có: chỉ nums[i] (độ dài 1) -> thêm nums[j]
                if length > dp[j].get(d, 0):           # max trên mọi i (không phụ thuộc thứ tự duyệt i)
                    dp[j][d] = length
                    if length > ans:
                        ans = length
        return ans


print([Solution().longestArithSeqLength(a) for a, _ in examples])
assert all(Solution().longestArithSeqLength(a) == w for a, w in examples)

[4, 3, 4, 3, 2]


## 10. Vì sao đúng

**(a) Cấu trúc tối ưu.** Gọi `f(j, d)` là độ dài lớn nhất của cấp số cộng công sai `d` kết thúc tại `j` (`≥ 1`, chính `nums[j]`). Một cấp số cộng độ dài `≥ 2` kết thúc tại `j` với phần tử trước là `nums[i]` (`i < j`, `nums[j] − nums[i] = d`) có phần đầu là một cấp số cộng công sai `d` kết thúc tại `i` (hoặc chỉ `nums[i]` khi độ dài tổng là `2`). Do đó `f(j, d) = 1 + max_{i<j, nums[j]−nums[i]=d} f(i, d)` (và `f(i, d) ≥ 1` luôn, tức độ dài tối thiểu `2`).

**(b) Tính `max` trên `i`.** Công thức đòi `max` trên mọi `i` thoả `nums[i] = nums[j] − d`. Bản ghi đè `dp[j][d] = dp[i].get(d,1)+1` theo `i` tăng dần cũng đúng: nếu `i₁ < i₂` cùng `nums[i₁] = nums[i₂] = nums[j] − d` thì `f(i₂, d) ≥ f(i₁, d)` (mọi cấp số cộng kết thúc tại `i₁` với phần tử cuối `nums[i₁] = nums[i₂]` có thể thay phần tử cuối bằng `nums[i₂]` vì `i₂ > i₁`: cùng giá trị, chỉ số lớn hơn ⇒ vẫn hợp lệ), nên giá trị ghi sau (với `i` lớn hơn) không nhỏ hơn. Dù vậy ta dùng `max` tường minh để không phụ thuộc lập luận này.

**(c) Đáp án.** `max_{j, d} f(j, d)` cũng bằng độ dài lớn nhất của dãy con cấp số cộng; với `n ≥ 2` luôn `≥ 2`.

**(d) Cách 3** là cùng DP với mảng. **Cách 4.** Cố định `d`, `best[v]` theo dõi độ dài lớn nhất của cấp số cộng công sai `d` **kết thúc bằng giá trị `v`** trong các phần tử đã duyệt; với `x` mới: `x − d` là phần tử trước. Độ dài `best[x−d] + 1` (nếu đã gặp) hoặc `1`. Khi `d = 0`, `x − d = x`: đọc `best[x]` **trước** khi ghi (ta tính `cand` trước rồi mới gán) nên đúng (`best[x]` cũ là độ dài kết thúc bằng giá trị `x` ở các lần gặp trước).

**(e) Cách 5.** Với cặp đầu tiên `(i, j)` cố định (hai phần tử đầu của dãy con) và công sai `d`, phần tử kế tiếp phải có giá trị `nums[j] + d` ở chỉ số `> j`; chọn chỉ số **nhỏ nhất** như vậy là tối ưu (lập luận trao đổi: chỉ số nhỏ hơn không làm mất lựa chọn nào sau đó). Lặp ⇒ độ dài tối đa của dãy bắt đầu bằng `(i, j)`.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle `2^n` trên **mọi** mảng `n ≤ 6` giá trị `0..4` và `n = 7` giá trị `0..3`; khớp nhau với `n = 1000`, giá trị tới `500` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi dãy con | `O(2^n·n)` | `O(n)` | oracle `n ≲ 14` |
| 2. DP `dict` | `O(n²)` | `O(n²)` | **đáp án chính** |
| 3. DP mảng `d+500` | `O(n²)` | `O(1001·n)` | nhanh hơn băm |
| 4. Mỗi công sai một lượt | `O(1001·n)` | `O(501)` | giá trị nhỏ |
| 5. Mở rộng tham lam | `O(n²·L)` | `O(n)` | `n` nhỏ |

Đo thực tế (`n = 1000`, giá trị tới `500`):

In [17]:
def bench(fn, arg, repeat=1):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1027)
big = [rng.randint(0, 500) for _ in range(1000)]
for name, fn in [("dict", arith_dict), ("array", arith_array), ("per_diff", arith_per_diff)]:
    print(f"{name:9s}: {bench(fn, big) * 1e3:8.1f} ms (n = 1000) -> {fn(big)}")
print(f"{'extend':9s}: {bench(arith_extend, big[:200]) * 1e3:8.1f} ms (n = 200) -> {arith_extend(big[:200])}")

dict     :    106.9 ms (n = 1000) -> 11


array    :     90.1 ms (n = 1000) -> 11


per_diff :    120.1 ms (n = 1000) -> 11
extend   :      3.5 ms (n = 200) -> 6


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle `2^n`, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `n = 1000`.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "dict": arith_dict,
    "array": arith_array,
    "per_diff": arith_per_diff,
    "extend": arith_extend,
    "brute": arith_brute,
    "Solution": lambda a: Solution().longestArithSeqLength(list(a)),
    "SolutionDict": lambda a: SolutionDict().longestArithSeqLength(list(a)),
    "SolutionArray": lambda a: SolutionArray().longestArithSeqLength(list(a)),
    "SolutionPerDiff": lambda a: SolutionPerDiff().longestArithSeqLength(list(a)),
    "SolutionExtend": lambda a: SolutionExtend().longestArithSeqLength(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().longestArithSeqLength(list(a)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(a, want=None, brute_n=13, extend_n=200):
    if want is None and len(a) <= brute_n:
        want = arith_brute(a)
    results = {}
    for name in IMPLS:
        if name in SLOW and len(a) > brute_n:
            continue
        if "xtend" in name and len(a) > extend_n:
            continue
        results[name] = IMPLS[name](a)
    assert len(set(results.values())) == 1, (a[:10], results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (a[:10], got, want)
    return got


for a, want in examples:
    check(a, want)
print("ví dụ đề: oracle 2^n và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle 2^n và 11 cài đặt đúng


### B. Edge case

- `n = 2`: luôn `2` (kể cả hai giá trị bằng nhau).
- Toàn giá trị bằng nhau: `n` (công sai `0`).
- Toàn cấp số cộng tăng / giảm.
- Cấp số cộng nằm rải rác (giữa các phần tử nhiễu).
- Hai cấp số cộng đan xen (công sai khác nhau).
- Công sai âm lớn: `[500, 0]`; công sai dương lớn: `[0, 500]`.
- Cấp số cộng dài nhất dùng công sai `0` nhưng có công sai khác dài bằng.

In [19]:
edge = [
    [1, 2], [5, 5], [0, 500], [500, 0], [7, 7, 7, 7, 7], [0, 1, 2, 3, 4, 5], [5, 4, 3, 2, 1, 0], [1, 9, 2, 8, 3, 7, 4], [0, 100, 50, 200, 100, 300, 150],
    [3, 3, 3, 6, 9, 12], [1, 5, 1, 5, 1, 5], [10, 7, 4, 1, 100, 200], [1, 3, 5, 7, 2, 4, 6, 8], [2, 4, 6, 3, 5, 7, 9], [0, 0, 1, 1, 2, 2], [0, 500, 0, 500, 0, 500],
]
for a in edge:
    check(a)
check(list(range(0, 500, 5)) * 1)
check([0] * 300)
print(len(edge) + 2, "edge case: mọi cài đặt khớp oracle (brute cho n ≤ 13)")
print("[0,0,1,1,2,2] ->", Solution().longestArithSeqLength([0, 0, 1, 1, 2, 2]), "| [3,3,3,6,9,12] ->", Solution().longestArithSeqLength([3, 3, 3, 6, 9, 12]))

18 edge case: mọi cài đặt khớp oracle (brute cho n ≤ 13)
[0,0,1,1,2,2] -> 3 | [3,3,3,6,9,12] -> 4


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_init_zero(nums):                                # dp mặc định 0 thay vì 1: độ dài thiếu 1
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            dp[j][d] = dp[i].get(d, 0) + 1
            ans = max(ans, dp[j][d])
    return ans


def bug_abs_diff(nums):                                 # dùng |hiệu| (trộn tăng và giảm)
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = abs(nums[j] - nums[i])
            dp[j][d] = dp[i].get(d, 1) + 1
            ans = max(ans, dp[j][d])
    return ans


def bug_subarray_only(nums):                            # chỉ dãy con LIÊN TIẾP
    best = cur = 2
    for k in range(2, len(nums)):
        if nums[k] - nums[k - 1] == nums[k - 1] - nums[k - 2]:
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best


def bug_only_positive(nums):                            # chỉ công sai dương
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            if d > 0:
                dp[j][d] = dp[i].get(d, 1) + 1
                ans = max(ans, dp[j][d])
    return ans


def bug_shared_by_diff(nums):                           # dp chỉ theo công sai (không theo chỉ số)
    dp = {}
    ans = 2
    for j in range(len(nums)):
        for i in range(j):
            d = nums[j] - nums[i]
            dp[d] = dp.get(d, 1) + 1
            ans = max(ans, dp[d])
    return ans


def bug_last_only(nums):                                # trả max của dp của phần tử cuối
    n = len(nums)
    dp = [dict() for _ in range(n)]
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            dp[j][d] = dp[i].get(d, 1) + 1
    return max(dp[-1].values())


def bug_per_diff_shared(nums):                          # không reset mảng giá trị giữa các công sai
    best = [0] * 501
    ans = 2
    for d in range(-500, 501):
        for x in nums:
            prev = x - d
            cand = best[prev] + 1 if 0 <= prev <= 500 and best[prev] else 1
            if cand > best[x]:
                best[x] = cand
            ans = max(ans, best[x])
    return ans


BUGS = [bug_init_zero, bug_abs_diff, bug_subarray_only, bug_only_positive, bug_shared_by_diff, bug_last_only, bug_per_diff_shared]
pool = [list(a) for n in range(2, 7) for a in itertools.product(range(4), repeat=n)]
for bug in BUGS:
    found = next((a for a in pool if bug(a) != arith_brute(a)), None)
    assert found is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên {found}: ra {bug(found)}, đúng {arith_brute(found)}")

bug_init_zero        sai trên [0, 0, 0]: ra 2, đúng 3
bug_abs_diff         sai trên [0, 1, 0]: ra 3, đúng 2
bug_subarray_only    sai trên [0, 0, 1, 0]: ra 2, đúng 3
bug_only_positive    sai trên [0, 0, 0]: ra 2, đúng 3
bug_shared_by_diff   sai trên [0, 0, 0]: ra 4, đúng 3
bug_last_only        sai trên [0, 0, 0, 1]: ra 2, đúng 3
bug_per_diff_shared  sai trên [0, 0]: ra 3, đúng 2


### D. Vét cạn mọi mảng nhỏ

Oracle `arith_brute` (duyệt mọi dãy con, định nghĩa). Mọi mảng độ dài `2..6` giá trị trong `0..4` (`25 + 125 + 625 + 3125 + 15625 = 19 525` mảng) và độ dài `7` giá trị trong `0..3` (`16 384` mảng). Với mỗi mảng mọi cài đặt phải khớp.

In [21]:
t0 = time.perf_counter()
cnt = 0
for hi_val, lengths in ((5, range(2, 7)), (4, [7])):
    for n in lengths:
        for arr in itertools.product(range(hi_val), repeat=n):
            a = list(arr)
            want = arith_brute(a)
            for name in IMPLS:
                assert IMPLS[name](a) == want, (name, a, IMPLS[name](a), want)
            cnt += 1
print(cnt, "mảng (n 2..6 trong 0..4; n = 7 trong 0..3):", len(IMPLS), "cài đặt khớp oracle 2^n;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 19525 + 16384

35909 mảng (n 2..6 trong 0..4; n = 7 trong 0..3): 11 cài đặt khớp oracle 2^n; 8.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách DP khớp nhau (n ≤ 300: mọi cài đặt nhanh; n lớn hơn: ba cài đặt chính); với `n ≤ 200` khớp `extend`; (2) `2 ≤ đáp án ≤ n`; (3) đảo mảng không đổi đáp án (đảo dãy con cấp số cộng vẫn là cấp số cộng); (4) `a → 500 − a` (đối xứng giá trị) không đổi; (5) cộng hằng (giữ trong `[0, 500]`) không đổi; (6) thêm một phần tử không làm đáp án giảm; (7) tồn tại dãy con cụ thể đạt đáp án (dựng lại và kiểm bằng `is_arith`).

In [22]:
def best_seq(nums):
    """Dựng một dãy con cấp số cộng dài nhất (danh sách chỉ số) từ DP có con trỏ cha."""
    n = len(nums)
    dp = [dict() for _ in range(n)]                    # dp[j][d] = (độ dài, chỉ số phần tử trước)
    best = (0, None, None)
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            prev_len = dp[i][d][0] if d in dp[i] else 1
            if d not in dp[j] or prev_len + 1 > dp[j][d][0]:
                dp[j][d] = (prev_len + 1, i)
            if dp[j][d][0] > best[0]:
                best = (dp[j][d][0], j, d)
    _, cur, d = best
    idx = [cur]
    while d in dp[cur]:
        cur = dp[cur][d][1]
        idx.append(cur)
    return sorted(idx)


rng = random.Random(20240116)
t0 = time.perf_counter()
for it in range(200):
    n = rng.choice([2, 3, 5, 10, 30, 100, 300, 300, 600])
    span = rng.choice([1, 3, 10, 50, 500])
    a = [rng.randint(0, span) for _ in range(n)]
    names = [k for k in fast if not ("xtend" in k and n > 200)] if n <= 300 else ["dict", "per_diff", "Solution"]
    res = {name: IMPLS[name](a) for name in names}
    assert len(set(res.values())) == 1, (n, res)                                        # (1)
    ans = res["Solution"]
    assert 2 <= ans <= n                                                                # (2)
    if n <= 300:
        assert Solution().longestArithSeqLength(a[::-1]) == ans                         # (3)
        assert Solution().longestArithSeqLength([max(a) - x for x in a]) == ans         # (4)
        if max(a) + 7 <= 500:
            assert Solution().longestArithSeqLength([x + 7 for x in a]) == ans          # (5)
        assert Solution().longestArithSeqLength(a + [rng.randint(0, span)]) >= ans      # (6)
        idx = best_seq(a)
        seq = [a[i] for i in idx]
        assert len(seq) == ans and is_arith(seq), (a[:8], idx, seq, ans)                # (7)
print("200 mảng ngẫu nhiên (n tới 600): 7 tính chất đúng (tính chất (3)-(7) cho n ≤ 300);", round(time.perf_counter() - t0, 1), "s")

200 mảng ngẫu nhiên (n tới 600): 7 tính chất đúng (tính chất (3)-(7) cho n ≤ 300); 6.4 s


In [23]:
# mảng nhỏ ngẫu nhiên (n <= 13): oracle 2^n cho mọi cài đặt, giá trị tới 500 và nhỏ
rng = random.Random(102701)
t0 = time.perf_counter()
for _ in range(600):
    n = rng.randint(2, 12)
    a = [rng.choice([rng.randint(0, 500), rng.randint(0, 6)]) for _ in range(n)]
    want = arith_brute(a)
    for name in IMPLS:
        assert IMPLS[name](a) == want, (name, a)
print("600 mảng nhỏ ngẫu nhiên: mọi cài đặt khớp oracle 2^n;", round(time.perf_counter() - t0, 1), "s")

600 mảng nhỏ ngẫu nhiên: mọi cài đặt khớp oracle 2^n; 2.5 s


### F. Giới hạn

`n = 1000`, giá trị tới `500`: toàn bằng nhau (`1000`), cấp số cộng dài (`0..499` lặp), ngẫu nhiên, hai giá trị xen kẽ; các cách DP khớp nhau.

In [24]:
t0 = time.perf_counter()
n = 1000
rng = random.Random(102702)
cases = [
    ([250] * n, n), ([i % 500 for i in range(n)], None), ([rng.randint(0, 500) for _ in range(n)], None), ([0, 500] * (n // 2), n // 2 + 0),
    ([(7 * i) % 501 for i in range(n)], None), ([min(500, i // 2) for i in range(n)], None),
]
for a, want in cases:
    res = {name: IMPLS[name](a) for name in ("dict", "array", "per_diff", "Solution", "SolutionDict", "SolutionArray", "SolutionPerDiff")}
    assert len(set(res.values())) == 1, res
    if want is not None:
        assert res["Solution"] == want, (res["Solution"], want)
    print(f"  đáp án = {res['Solution']}")
print("n = 1000: các DP khớp nhau;", round(time.perf_counter() - t0, 2), "s")

  đáp án = 1000


  đáp án = 500


  đáp án = 10


  đáp án = 500


  đáp án = 72


  đáp án = 500
n = 1000: các DP khớp nhau; 3.53 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: khởi tạo `dp` mặc định `0` (độ dài thiếu 1)

In [25]:
print("bug:", bug_init_zero([3, 6, 9, 12]), "| đúng:", Solution().longestArithSeqLength([3, 6, 9, 12]))
assert bug_init_zero([3, 6, 9, 12]) == 3 and Solution().longestArithSeqLength([3, 6, 9, 12]) == 4

bug: 3 | đúng: 4


**Vì sao sai:** cặp `(i, j)` đã là cấp số cộng độ dài `2`; `dp[i].get(d, 1) + 1 = 2` (mặc định `1` = chỉ có `nums[i]`). Mặc định `0` làm mọi độ dài thiếu `1`.

### Lỗi 2: dùng giá trị tuyệt đối của hiệu

In [26]:
print("bug:", bug_abs_diff([1, 3, 1]), "| đúng:", Solution().longestArithSeqLength([1, 3, 1]))
assert bug_abs_diff([1, 3, 1]) == 3 and Solution().longestArithSeqLength([1, 3, 1]) == 2

bug: 3 | đúng: 2


**Vì sao sai:** `[1, 3, 1]` có hiệu `+2` rồi `−2`: **không** là cấp số cộng. Dấu của công sai quan trọng.

### Lỗi 3: nhầm dãy con với dãy con liên tiếp

In [27]:
print("bug:", bug_subarray_only([9, 4, 7, 2, 10]), "| đúng:", Solution().longestArithSeqLength([9, 4, 7, 2, 10]))
assert bug_subarray_only([9, 4, 7, 2, 10]) == 2 and Solution().longestArithSeqLength([9, 4, 7, 2, 10]) == 3

bug: 2 | đúng: 3


**Vì sao sai:** đề cho phép **xoá** phần tử (dãy con), nên `[4, 7, 10]` hợp lệ dù không liên tiếp.

### Lỗi 4: chỉ xét công sai dương

In [28]:
print("bug:", bug_only_positive([5, 4, 3, 2, 1]), "| đúng:", Solution().longestArithSeqLength([5, 4, 3, 2, 1]))
assert bug_only_positive([5, 4, 3, 2, 1]) == 2 and Solution().longestArithSeqLength([5, 4, 3, 2, 1]) == 5

bug: 2 | đúng: 5


**Vì sao sai:** công sai có thể **âm** hoặc **bằng 0**.

### Lỗi 5: `dp` chỉ theo công sai (không theo chỉ số cuối)

In [29]:
bad = next(a for a in pool if bug_shared_by_diff(a) != arith_brute(a))
print("phản ví dụ:", bad, "| bug:", bug_shared_by_diff(bad), "| đúng:", arith_brute(bad))
assert bug_shared_by_diff(bad) != arith_brute(bad)

phản ví dụ: [0, 0, 0] | bug: 4 | đúng: 3


**Vì sao sai:** hai cặp khác nhau cùng công sai nhưng không nối với nhau bị cộng dồn. Trạng thái phải gồm **chỉ số cuối** để chắc chắn phần tử trước nối đúng.

### Lỗi 6 (bonus): chỉ lấy max của `dp` ở phần tử cuối, hoặc không reset giữa các công sai

In [30]:
a = [3, 6, 9, 12, 0]
print("chỉ phần tử cuối:", bug_last_only(a), "| đúng:", Solution().longestArithSeqLength(a))
bad = next(a for a in pool if bug_per_diff_shared(a) != arith_brute(a))
print("không reset mảng:", bad, bug_per_diff_shared(bad), "| đúng:", arith_brute(bad))
assert bug_last_only(a) != Solution().longestArithSeqLength(a) and bug_per_diff_shared(bad) != arith_brute(bad)

chỉ phần tử cuối: 2 | đúng: 4
không reset mảng: [0, 0] 3 | đúng: 2


**Vì sao sai:** dãy dài nhất không nhất thiết kết thúc ở phần tử cuối; còn với cách "mỗi công sai một lượt", mảng giá trị phải được **đặt lại** cho mỗi công sai.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "dãy con dài nhất với quan hệ giữa hai phần tử liên tiếp phụ thuộc một tham số" → DP theo **(phần tử cuối, tham số)**.

**Mẹo:**

1. Trạng thái `dp[j][d]` với `d` là công sai; mặc định `dp[i].get(d, 1)`.
2. Giá trị nhỏ ⇒ có thể lặp theo từng `d` với mảng theo giá trị (`O(1001·n)`).
3. Công sai `0` hợp lệ (phần tử trùng nhau).
4. Dùng `max` trên `i`, và `ans` khởi tạo `2`.
5. Oracle `2^n` theo định nghĩa trên mảng rất nhỏ.
6. Dựng lại dãy bằng con trỏ cha để kiểm chứng.

**Họ bài cùng pattern:** Longest Arithmetic Subsequence of Given Difference (1218), Arithmetic Slices II - Subsequence (446), Longest Increasing Subsequence (300), Longest Fibonacci-Like Subsequence (873), Number of Longest Increasing Subsequence (673), Length of Longest Fibonacci Subsequence.

## 15. Biến thể

1. **Công sai cho trước** `d` (LeetCode 1218): `O(n)` bằng `dict` theo giá trị.
2. **Dãy con cấp số cộng liên tiếp** (subarray): một lượt `O(n)`.
3. **Trả về dãy con** (đã làm ở `best_seq` mục 12E).

Cài đặt và đối chiếu brute:

In [31]:
def longest_given_diff(nums, d):
    best = {}
    ans = 0
    for x in nums:
        best[x] = best.get(x - d, 0) + 1              # đọc best[x - d] trước khi ghi best[x] (d = 0 cũng đúng)
        ans = max(ans, best[x])
    return ans


def longest_given_diff_brute(nums, d):
    n = len(nums)
    ans = 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if all(seq[k + 1] - seq[k] == d for k in range(len(seq) - 1)):
            ans = max(ans, len(seq))
    return ans


def longest_arith_subarray(nums):
    best = cur = min(2, len(nums))
    for k in range(2, len(nums)):
        cur = cur + 1 if nums[k] - nums[k - 1] == nums[k - 1] - nums[k - 2] else 2
        best = max(best, cur)
    return best


def subarray_brute(nums):
    n, best = len(nums), min(2, len(nums))
    for i in range(n):
        for j in range(i + 1, n):
            if is_arith(nums[i:j + 1]):
                best = max(best, j - i + 1)
    return best


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(2, 11)
    a = [rng.randint(0, 6) for _ in range(n)]
    d = rng.randint(-3, 3)
    assert longest_given_diff(a, d) == longest_given_diff_brute(a, d)                    # (1)
    assert longest_arith_subarray(a) == subarray_brute(a)                                # (2)
    assert longest_arith_subarray(a) <= Solution().longestArithSeqLength(a)
print("công sai cho trước, dãy con liên tiếp: khớp oracle; (liên tiếp ≤ dãy con)")

công sai cho trước, dãy con liên tiếp: khớp oracle; (liên tiếp ≤ dãy con)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy con | định nghĩa | `O(2^n·n)` | `O(n)` | oracle |
| 2. DP `dict` | `(cuối, công sai)` | `O(n²)` | `O(n²)` | **đáp án chính** |
| 3. DP mảng `d+500` | như trên, mảng | `O(n²)` | `O(1001n)` | nhanh hơn băm |
| 4. Mỗi công sai một lượt | mảng theo giá trị | `O(1001n)` | `O(501)` | giá trị nhỏ |
| 5. Mở rộng tham lam | từ mỗi cặp | `O(n²L)` | `O(n)` | `n` nhỏ |

Mọi cách khớp oracle `2^n` trên `35 909` mảng nhỏ và khớp nhau tới `n = 1000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print(Solution().longestArithSeqLength([3, 6, 9, 12]), Solution().longestArithSeqLength([9, 4, 7, 2, 10]), Solution().longestArithSeqLength([20, 1, 15, 3, 10, 5, 8]),
      Solution().longestArithSeqLength([1, 1, 1]), Solution().longestArithSeqLength([1, 2]))

4 3 4 3 2


### Bài 2 (tính tay)

`nums = [1, 5, 3, 7, 9]`. (a) Liệt kê các cấp số cộng dài `3` (theo chỉ số). (b) Có cấp số cộng dài `4` không? (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP `dict`:

In [33]:
def dict_todo(nums):
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            pass        # TODO 1: dp[j][d] = max(dp[j].get(d, 0), dp[i].get(d, 1) + 1)
            pass        # TODO 2: ans = max(ans, dp[j][d])
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(dict_todo(a) == w for a, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mỗi công sai một lượt với mảng theo giá trị** (nhớ đọc `best[x − d]` trước khi ghi `best[x]`):

In [34]:
class MySolution:
    def longestArithSeqLength(self, nums):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().longestArithSeqLength(list(a)) == w for a, w in examples)

### Bài 5 (mini challenge): công sai cho trước (1218)

Cho `arr` và số nguyên `difference`, trả về độ dài dãy con cấp số cộng dài nhất có công sai **đúng bằng** `difference`. Gợi ý: `dp[x] = dp[x − difference] + 1`, đọc `dp[x − difference]` trước khi ghi `dp[x]`. Kiểm bằng brute `2^n`.

In [35]:
def longest_subsequence(arr, difference):
    pass            # TODO


# Khi làm xong:
# assert longest_subsequence([1, 2, 3, 4], 1) == 4 and longest_subsequence([1, 5, 7, 8, 5, 3, 4, 2, 1], -2) == 4

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tìm công sai chung dài nhất.
- Mức 2: `[1,1,1]` có công sai `0` ⇒ `3`; `[20,1,15,3,10,5,8]` có `[20,15,10,5]`.
- Mức 3: `4 3 4 3 2`.

**Bài 2**

- Mức 1: `[1, 5, 9]` công sai `4`; `[5, 7, 9]` công sai `2`; `[1, 3, 5]`? `3` đứng sau `5` nên không.
- Mức 2: không có cấp số cộng dài `4`.
- Mức 3: đáp án `3` (`[1,5,9]` và `[5,7,9]`).

**Bài 3**

- Mức 1: `dp[j][d]` lấy max trên `i`.
- Mức 2: `dp[i].get(d, 1) + 1` là ứng viên.
- Mức 3: `dp[j][d] = max(dp[j].get(d, 0), dp[i].get(d, 1) + 1)`; `ans = max(ans, dp[j][d])`.

**Bài 4**

- Mức 1: `for d in range(-500, 501): best = [0] * 501`.
- Mức 2: `prev = x - d`; ứng viên `best[prev] + 1` nếu `0 <= prev <= 500` và `best[prev] > 0`, ngược lại `1`.
- Mức 3: `best[x] = max(best[x], cand)`; `ans = max(ans, best[x])`.

**Bài 5**

- Mức 1: `dp = {}`.
- Mức 2: `dp[x] = dp.get(x - difference, 0) + 1`.
- Mức 3: trả `max(dp.values())`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = (Solution().longestArithSeqLength([3, 6, 9, 12]), Solution().longestArithSeqLength([9, 4, 7, 2, 10]), Solution().longestArithSeqLength([20, 1, 15, 3, 10, 5, 8]),
        Solution().longestArithSeqLength([1, 1, 1]), Solution().longestArithSeqLength([1, 2]))
print(vals)
assert vals == (4, 3, 4, 3, 2)
print("Bài 1 OK")

(4, 3, 4, 3, 2)
Bài 1 OK


### Bài 2

In [37]:
a = [1, 5, 3, 7, 9]
triples = [(i, j, k) for i, j, k in itertools.combinations(range(5), 3) if a[j] - a[i] == a[k] - a[j]]
quads = [c for c in itertools.combinations(range(5), 4) if is_arith([a[i] for i in c])]
print([[a[i] for i in t] for t in triples], quads)
assert [[a[i] for i in t] for t in triples] == [[1, 5, 9], [5, 7, 9]] and quads == []
assert Solution().longestArithSeqLength(a) == 3 == arith_brute(a)
print("Bài 2 OK")

[[1, 5, 9], [5, 7, 9]] []
Bài 2 OK


### Bài 3

In [38]:
def dict_answer(nums):
    n = len(nums)
    dp = [dict() for _ in range(n)]
    ans = 2
    for j in range(n):
        for i in range(j):
            d = nums[j] - nums[i]
            dp[j][d] = max(dp[j].get(d, 0), dp[i].get(d, 1) + 1)
            ans = max(ans, dp[j][d])
    return ans


assert all(dict_answer(a) == w for a, w in examples)
rng = random.Random(3)
for _ in range(400):
    a = [rng.randint(0, 6) for _ in range(rng.randint(2, 11))]
    assert dict_answer(a) == arith_brute(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def longestArithSeqLength(self, nums):
        ans = 2
        for d in range(-500, 501):
            best = [0] * 501
            for x in nums:
                prev = x - d
                cand = best[prev] + 1 if 0 <= prev <= 500 and best[prev] > 0 else 1
                if cand > best[x]:
                    best[x] = cand
                    ans = max(ans, cand)
        return ans


assert all(MySolutionAnswer().longestArithSeqLength(list(a)) == w for a, w in examples)
rng = random.Random(4)
for _ in range(400):
    a = [rng.randint(0, 6) for _ in range(rng.randint(2, 11))]
    assert MySolutionAnswer().longestArithSeqLength(a) == arith_brute(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def longest_subsequence(arr, difference):
    dp = {}
    for x in arr:
        dp[x] = dp.get(x - difference, 0) + 1
    return max(dp.values())


assert longest_subsequence([1, 2, 3, 4], 1) == 4 and longest_subsequence([1, 5, 7, 8, 5, 3, 4, 2, 1], -2) == 4
rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(0, 6) for _ in range(rng.randint(1, 11))]
    d = rng.randint(-3, 3)
    assert longest_subsequence(a, d) == longest_given_diff_brute(a, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (dãy con dài nhất với tham số quan hệ giữa hai phần tử liên tiếp → DP theo `(cuối, tham số)`):**

```python
class Solution:
    def longestArithSeqLength(self, nums: List[int]) -> int:
        n = len(nums)
        dp = [{} for _ in range(n)]
        ans = 2
        for j in range(n):
            for i in range(j):
                d = nums[j] - nums[i]
                length = dp[i].get(d, 1) + 1
                if length > dp[j].get(d, 0):
                    dp[j][d] = length
                    ans = max(ans, length)
        return ans
```

**Checklist trước khi submit:**

- [ ] Mặc định `dp[i].get(d, 1)` (không phải `0`); `ans` khởi tạo `2`?
- [ ] Công sai **có dấu** (không `abs`); gồm `0` và số âm?
- [ ] Dãy **con** (không cần liên tiếp)?
- [ ] Trạng thái có **chỉ số cuối** (không chỉ công sai)?
- [ ] Trả max toàn cục (không chỉ phần tử cuối)?
- [ ] Đã thử `[3,6,9,12]`, `[9,4,7,2,10]`, `[20,1,15,3,10,5,8]`, `[1,1,1]`?
- [ ] Đúng tên `class Solution`, `longestArithSeqLength(self, nums)`, không in debug.